In [ ]:
"""
Tredence Case Study: The Self-Pruning Neural Network
Author: [Your Name]
Role: AI Engineering Intern - 2025 Cohort
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import math

# ==============================================================
# 1. The Prunable Linear Layer
# ==============================================================
class PrunableLinear(nn.Module):
    """
    A custom linear layer where each weight is dynamically scaled by a
    learnable gate. The gate values are restricted between 0 and 1 via Sigmoid.
    """
    def __init__(self, in_features, out_features):
        super(PrunableLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features

        # Standard learnable weight and bias
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.bias = nn.Parameter(torch.Tensor(out_features))

        # The secondary 'gate_score' tensor. It must be identical in shape to the weight.
        self.gate_scores = nn.Parameter(torch.Tensor(out_features, in_features))

        self.reset_parameters()

    def reset_parameters(self):
        """Initializes weights using standard PyTorch Kaiming uniform."""
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(self.bias, -bound, bound)

        # IMPORTANT: Initialize gate_scores to 3.0.
        # torch.sigmoid(3.0) ≈ 0.95. This ensures all gates start "fully open".
        # If initialized around 0, gates would be ~0.5, harming early convergence.
        nn.init.constant_(self.gate_scores, 3.0)

    def forward(self, x):
        # Pass gate_scores through a sigmoid so values strictly fall between 0 and 1
        gates = torch.sigmoid(self.gate_scores)

        # Element-wise multiply standard weights by the gating values
        pruned_weights = self.weight * gates

        # Proceed with standard affine transform using the pruned weights
        return F.linear(x, pruned_weights, self.bias)

# ==============================================================
# 2. The Neural Network
# ==============================================================
class PrunableNet(nn.Module):
    """Simple Feed-Forward Multi-Layer Perceptron (MLP) for CIFAR-10."""
    def __init__(self):
        super(PrunableNet, self).__init__()
        self.flatten = nn.Flatten()

        # CIFAR-10 dims: 3 channels * 32 * 32 pixels = 3072 input features
        self.fc1 = PrunableLinear(3072, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 10) # 10 classes output

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x) # CrossEntropyLoss handles the softmax internally
        return x

# ==============================================================
# 3. Regularization & Evaluation Utilities
# ==============================================================
def get_sparsity_loss(model):
    """
    Computes the Sparsity Loss.
    Because sigmoid(x) > 0, the L1 Norm (sum of absolute values)
    is equivalent to the raw sum of all gates across the network.
    """
    sparsity_loss = 0.0
    for module in model.modules():
        if isinstance(module, PrunableLinear):
            gates = torch.sigmoid(module.gate_scores)
            sparsity_loss += torch.sum(gates)
    return sparsity_loss

def calculate_sparsity_level(model, threshold=1e-2):
    """
    Calculates what percentage of total network gates fall below a certain threshold.
    """
    pruned_count = 0
    total_count = 0
    with torch.no_grad():
        for module in model.modules():
            if isinstance(module, PrunableLinear):
                gates = torch.sigmoid(module.gate_scores)
                # Count elements strictly smaller than the 1e-2 threshold
                pruned_count += torch.sum(gates < threshold).item()
                total_count += gates.numel()

    # Avoid division by zero
    if total_count == 0:
        return 0.0
    return (pruned_count / total_count) * 100

# ==============================================================
# 4. Core Training Pipeline
# ==============================================================
def train_model(lmbda, epochs, device='cpu'):
    print(f"\n[{'='*40}]")
    print(f"--- Training with Sparsity Lambda = {lmbda} ---")
    print(f"[{'='*40}]")

    # Basic Data Augmentation & Normalization
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    # Download/Load CIFAR10
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    trainloader = torch.utils.data.DataLoader(trainset, batch_size=256, shuffle=True, num_workers=2)

    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    testloader = torch.utils.data.DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

    model = PrunableNet().to(device)
    criterion = nn.CrossEntropyLoss()

    # lr is slightly higher than default to help push through vanishing sigmoid gradients
    optimizer = optim.Adam(model.parameters(), lr=0.003)

    # --- Training Loop ---
    for epoch in range(epochs):
        model.train()
        running_ce_loss = 0.0
        running_total_loss = 0.0

        for inputs, labels in trainloader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            # Forward Pass
            outputs = model(inputs)
            cls_loss = criterion(outputs, labels)

            # Compute Custom Regularization
            sparsity_loss = get_sparsity_loss(model)

            # Formulate Total Loss
            total_loss = cls_loss + (lmbda * sparsity_loss)

            # Backward & Step
            total_loss.backward()
            optimizer.step()

            running_ce_loss += cls_loss.item()
            running_total_loss += total_loss.item()

        print(f"Epoch {epoch+1:02d}/{epochs} | Total Loss: {running_total_loss/len(trainloader):.4f} | Base Loss: {running_ce_loss/len(trainloader):.4f}")

    # --- Evaluation Phase ---
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    final_accuracy = 100.0 * correct / total
    final_sparsity = calculate_sparsity_level(model, threshold=1e-2)

    print(f"\n=> Test Accuracy: {final_accuracy:.2f}%")
    print(f"=> Final Sparsity (<0.01): {final_sparsity:.2f}%")

    return model, final_accuracy, final_sparsity


# ==============================================================
# 5. Script Execution & Output Plotting
# ==============================================================
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n>>> Script initializing. Backend Device: {device} <<<\n")

    # Evaluate 3 different magnitudes of the penalty parameter lambda
    test_lambdas = [0.0, 1e-5, 1e-4]
    training_epochs = 20 # Increased epochs ensures gates converge properly

    experiment_results = []
    models_dict = {}

    for lam in test_lambdas:
        trained_model, test_acc, model_sparsity = train_model(lam, epochs=training_epochs, device=device)
        experiment_results.append((lam, test_acc, model_sparsity))
        models_dict[lam] = trained_model

    # 1. Print Summary Table
    print("\n\n" + "="*50)
    print("                 EXPERIMENT SUMMARY")
    print("="*50)
    print(f"{'Lambda (L1 Penalty)':<22} | {'Test Accuracy (%)':<18} | {'Sparsity Level (%)'}")
    print("-" * 65)
    for res in experiment_results:
        # e.g., '1e-05' to avoid scientific notation float misalignment
        lam_str = f"{res[0]:.0e}" if res[0] > 0 else "0.0"
        print(f"{lam_str:<22} | {res[1]:<18.2f} | {res[2]:.2f}%")
    print("=" * 65 + "\n")

    # 2. Extract Data for Plotting (Use the most heavily penalized model to see distinct cluster)
    best_lambda = test_lambdas[-1]
    target_model = models_dict[best_lambda]

    all_final_gates = []
    with torch.no_grad():
        for module in target_model.modules():
            if isinstance(module, PrunableLinear):
                # Flatten the 2D tensor of gates into a 1D array for histogram
                layer_gates = torch.sigmoid(module.gate_scores).cpu().numpy().flatten()
                all_final_gates.extend(layer_gates)

    # 3. Generate Matplotlib visualization
    plt.figure(figsize=(10, 6))

    # We use a logarithmic y-scale because the spike at 0 is MASSIVE (millions of parameters).
    # Without log scale, you cannot see the cluster of active connections near 1.
    counts, bins, patches = plt.hist(all_final_gates, bins=50, color='#3498db', edgecolor='black', alpha=0.85)

    plt.title(f"Self-Pruning Network: Gate Value Distribution\n($\lambda$ = {best_lambda})", fontsize=14, pad=15)
    plt.xlabel("Gate Values after Sigmoid Transformation (0=Pruned, 1=Active)", fontsize=12)
    plt.ylabel("Parameter Count (Log Scale)", fontsize=12)

    plt.yscale('log')
    plt.grid(True, axis='y', alpha=0.3, linestyle='--')

    # Format and save
    plt.tight_layout()
    output_filename = "gate_distribution.png"
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n>>> Target plot generated and saved successfully as '{output_filename}'! <<<")